# Análise exploratória e classificação do tipo de vinho

**Disciplina:** Machine Learning / Ciência de Dados  
**Aluno(a):** *Preencha com seu nome*  
**Dataset:** Wine Quality — UCI Machine Learning Repository  
**Fonte oficial:** https://archive.ics.uci.edu/dataset/186/wine+quality

## Objetivo

Este notebook investiga as diferenças químicas entre vinhos **tintos** e **brancos** do conjunto *Wine Quality* da UCI e constrói classificadores para prever o seu tipo a partir de medições físico-químicas. A atividade contempla análise exploratória, perguntas orientadoras, manipulação com Pandas/NumPy/Matplotlib, divisão treino–teste, KNN, Random Forest, métricas comparáveis e persistência do melhor modelo em `joblib`.

> **Por que este dataset atende à proposta intermediária?** A UCI disponibiliza dois arquivos separados — um de vinho tinto e outro de vinho branco — e não uma coluna `target` única. Portanto, a variável-alvo `tipo_vinho` é criada de forma rastreável a partir da origem de cada arquivo. O objetivo não é prever a nota `quality`; ela é tratada apenas como uma variável descritiva e fica fora das entradas dos modelos para evitar que uma avaliação sensorial interfira na classificação química.

## Perguntas e hipóteses

As perguntas abaixo guiam toda a exploração. Elas são respondidas com estatísticas e gráficos antes da modelagem.

1. **H1 — Acidez volátil:** vinhos tintos apresentam, em média, maior acidez volátil que vinhos brancos?
2. **H2 — Açúcar e dióxido de enxofre:** vinhos brancos têm maiores níveis de açúcar residual e de dióxido de enxofre total?
3. **H3 — Nota sensorial:** a distribuição da nota `quality` e a proporção de vinhos com nota maior ou igual a 6 diferem entre os tipos?
4. **H4 — Predição:** as 11 medidas físico-químicas, sem usar `quality`, distinguem tintos de brancos com acurácia superior a 85% em dados nunca vistos? KNN ou Random Forest se comporta melhor?

As hipóteses H1–H3 são descritivas; H4 é avaliada com um conjunto de teste separado e com métricas de classificação.

In [ ]:
# Reprodutibilidade e bibliotecas
from pathlib import Path
from urllib.request import urlretrieve
from zipfile import ZipFile
import warnings

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    accuracy_score,
    classification_report,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import GridSearchCV, StratifiedKFold, train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import LabelEncoder, StandardScaler

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
warnings.filterwarnings("ignore", category=FutureWarning)

plt.style.use("seaborn-v0_8-whitegrid")
pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda x: f"{x:,.3f}")

In [ ]:
# Download reprodutível da fonte oficial (executado somente se os CSVs ainda não existirem).
ROOT_DIR = Path.cwd()
DATA_DIR = ROOT_DIR / "data"
DATA_DIR.mkdir(exist_ok=True)

red_path = DATA_DIR / "winequality-red.csv"
white_path = DATA_DIR / "winequality-white.csv"
source_url = "https://archive.ics.uci.edu/static/public/186/wine+quality.zip"

if not (red_path.exists() and white_path.exists()):
    zip_path = DATA_DIR / "wine_quality_uci.zip"
    print("Baixando os dados oficiais da UCI...")
    urlretrieve(source_url, zip_path)
    with ZipFile(zip_path) as zf:
        zf.extractall(DATA_DIR)

red = pd.read_csv(red_path, sep=";")
white = pd.read_csv(white_path, sep=";")

# Criação explícita e auditável da variável-alvo: ela vem do arquivo de origem.
red["tipo_vinho"] = "tinto"
white["tipo_vinho"] = "branco"
df_original = pd.concat([red, white], ignore_index=True)

print(f"Observações originais: {df_original.shape[0]:,}")
print(f"Variáveis: {df_original.shape[1]}")
df_original.head()

## 1. Auditoria e limpeza dos dados

Antes de responder às hipóteses, verifico tipos, ausências e repetições. Registros inteiramente duplicados são removidos **antes** da separação treino–teste; assim, uma cópia idêntica não pode aparecer simultaneamente em treino e teste e elevar artificialmente a métrica. Não removo outliers automaticamente: em vinho, valores extremos podem representar produtos reais e a decisão exigiria conhecimento de domínio adicional.

In [ ]:
auditoria = pd.DataFrame({
    "tipo": df_original.dtypes.astype(str),
    "ausentes": df_original.isna().sum(),
    "unicos": df_original.nunique(),
})
display(auditoria)

duplicados = int(df_original.duplicated().sum())
print(f"Linhas completamente duplicadas: {duplicados:,}")

df = df_original.drop_duplicates().copy()
print(f"Formato após remoção de duplicados: {df.shape[0]:,} linhas × {df.shape[1]} colunas")
print("\nDistribuição da variável-alvo:")
display(pd.DataFrame({
    "quantidade": df["tipo_vinho"].value_counts(),
    "percentual": df["tipo_vinho"].value_counts(normalize=True).mul(100),
}).round(2))

In [ ]:
# Resumo numérico separado por tipo de vinho.
resumo = df.groupby("tipo_vinho").agg(["mean", "median", "std", "min", "max"])
display(resumo.T.round(3))

## 2. Exploração guiada por hipóteses

A comparação de média e mediana evita concluir apenas por uma medida: a média revela o nível global, enquanto a mediana é mais resistente a valores extremos. Os gráficos apresentam a mesma evidência visualmente.

In [ ]:
# H1 — Acidez volátil: comparação de centro e dispersão.
h1 = df.groupby("tipo_vinho")["volatile acidity"].agg(["count", "mean", "median", "std"])
display(h1.round(3))

fig, ax = plt.subplots(figsize=(8, 4.5))
ordem = ["tinto", "branco"]
ax.boxplot(
    [df.loc[df.tipo_vinho.eq(t), "volatile acidity"] for t in ordem],
    tick_labels=ordem,
    patch_artist=True,
    boxprops={"facecolor": "#B03A2E"},
    medianprops={"color": "black", "linewidth": 2},
)
ax.set_title("H1 — Acidez volátil por tipo de vinho")
ax.set_xlabel("Tipo de vinho")
ax.set_ylabel("Acidez volátil")
plt.show()

diferenca_h1 = h1.loc["tinto", "mean"] - h1.loc["branco", "mean"]
print(f"Diferença das médias (tinto − branco): {diferenca_h1:.3f}")
print("Conclusão H1:", "sustentada" if diferenca_h1 > 0 else "não sustentada", "pelos dados.")

In [ ]:
# H2 — Açúcar residual e dióxido de enxofre total.
colunas_h2 = ["residual sugar", "total sulfur dioxide"]
h2 = df.groupby("tipo_vinho")[colunas_h2].agg(["mean", "median"])
display(h2.round(3))

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
cores = ["#B03A2E", "#F4D03F"]
for ax, coluna in zip(axes, colunas_h2):
    medianas = df.groupby("tipo_vinho")[coluna].median().reindex(["tinto", "branco"])
    barras = ax.bar(medianas.index, medianas.values, color=cores)
    ax.bar_label(barras, labels=[f"{v:.1f}" for v in medianas.values], padding=3)
    ax.set_title(f"H2 — Mediana de {coluna}")
    ax.set_ylabel(coluna)
fig.suptitle("Vinhos brancos concentram mais açúcar e SO2 total?", y=1.02, fontsize=13)
plt.tight_layout()
plt.show()

for coluna in colunas_h2:
    delta = df.loc[df.tipo_vinho.eq("branco"), coluna].median() - df.loc[df.tipo_vinho.eq("tinto"), coluna].median()
    print(f"{coluna}: diferença da mediana (branco − tinto) = {delta:.2f}")

In [ ]:
# H3 — Nota de qualidade: comparação proporcional, não usada como preditora.
tabela_quality = pd.crosstab(df["quality"], df["tipo_vinho"], normalize="columns").mul(100)
display(tabela_quality.round(2))

fig, ax = plt.subplots(figsize=(9, 5))
tabela_quality.plot(kind="bar", ax=ax, color=["#F4D03F", "#B03A2E"])
ax.set_title("H3 — Distribuição percentual da nota quality por tipo")
ax.set_xlabel("Nota quality")
ax.set_ylabel("Percentual dentro do tipo (%)")
ax.legend(title="Tipo")
plt.xticks(rotation=0)
plt.show()

prop_boa = (df.assign(qualidade_ge_6=df["quality"].ge(6))
              .groupby("tipo_vinho")["qualidade_ge_6"].mean()
              .mul(100))
print("Percentual de vinhos com quality ≥ 6:")
display(prop_boa.rename("percentual").round(2).to_frame())
print("Observação: esta diferença é descritiva; `quality` não entra no modelo de classificação.")

In [ ]:
# Relação entre medidas: correlação das variáveis químicas e quality.
variaveis_numericas = df.select_dtypes(include="number").columns.tolist()
corr = df[variaveis_numericas].corr(numeric_only=True)

fig, ax = plt.subplots(figsize=(10, 8))
imagem = ax.imshow(corr, cmap="coolwarm", vmin=-1, vmax=1)
ax.set_xticks(range(len(corr.columns)), corr.columns, rotation=75, ha="right", fontsize=8)
ax.set_yticks(range(len(corr.index)), corr.index, fontsize=8)
ax.set_title("Correlação de Pearson entre variáveis numéricas")
fig.colorbar(imagem, ax=ax, label="Correlação")
plt.tight_layout()
plt.show()

pares = (corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))
           .stack()
           .sort_values(key=np.abs, ascending=False)
           .head(8))
print("Pares de variáveis com maior correlação absoluta:")
display(pares.rename("correlação").to_frame().round(3))

## 3. Preparação para modelagem

As entradas são somente as **11 medidas físico-químicas**. A coluna `quality` é deliberadamente excluída porque é uma nota sensorial fornecida pela base, não uma medição química disponível no mesmo processo de classificação.

A divisão usada é **80% para treino e 20% para teste**, com `stratify=y`. O treino de 80% fornece observações suficientes para ajustar e selecionar hiperparâmetros; os 20% restantes formam uma avaliação final independente. A estratificação preserva no teste a proporção tinto/branco, importante porque há mais brancos na base.

In [ ]:
feature_cols = [
    coluna for coluna in df.columns
    if coluna not in ["tipo_vinho", "quality"]
]
X = df[feature_cols].copy()

encoder = LabelEncoder()
y = encoder.fit_transform(df["tipo_vinho"])
mapa_classes = dict(zip(encoder.classes_, encoder.transform(encoder.classes_)))

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=y,
)

print(f"Features usadas ({len(feature_cols)}): {feature_cols}")
print(f"Mapeamento da variável-alvo: {mapa_classes}")
print(f"Treino: {X_train.shape[0]:,} observações ({len(X_train) / len(X):.0%})")
print(f"Teste:  {X_test.shape[0]:,} observações ({len(X_test) / len(X):.0%})")
print("\nProporção por classe — treino e teste:")
display(pd.DataFrame({
    "treino": pd.Series(y_train).map(dict(enumerate(encoder.classes_))).value_counts(normalize=True),
    "teste": pd.Series(y_test).map(dict(enumerate(encoder.classes_))).value_counts(normalize=True),
}).mul(100).round(2))

## 4. Modelos escolhidos

- **KNN:** classifica por proximidade. Como ele depende de distância, é obrigatório padronizar as escalas; por isso o `StandardScaler` está dentro do `Pipeline`, ajustado apenas no treino em cada dobra de validação.
- **Random Forest:** combina árvores de decisão em um conjunto; captura relações não lineares e não exige padronização. A importância das variáveis ajuda a interpretar o modelo.

Para o KNN, comparo valores de `k` em uma validação cruzada estratificada de 3 dobras apenas no conjunto de treino. A distância Euclidiana e o peso por distância foram mantidos por serem escolhas coerentes para medições padronizadas. O conjunto de teste continua intocado até a comparação final.

In [ ]:
# KNN: pipeline evita vazamento de escala do conjunto de teste.
knn_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("knn", KNeighborsClassifier()),
])
cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)
grade_knn = {
    "knn__n_neighbors": [5, 9, 15, 21],
    "knn__weights": ["distance"],
    "knn__p": [2],
}
busca_knn = GridSearchCV(
    estimator=knn_pipeline,
    param_grid=grade_knn,
    scoring="accuracy",
    cv=cv,
    n_jobs=1,
    return_train_score=False,
)
busca_knn.fit(X_train, y_train)

print("Melhores hiperparâmetros do KNN:", busca_knn.best_params_)
print(f"Melhor acurácia média na validação cruzada estratificada (3 dobras): {busca_knn.best_score_:.2%}")

In [ ]:
# Random Forest: configuração fixa, reprodutível e com paralelismo.
random_forest = RandomForestClassifier(
    n_estimators=250,
    max_features="sqrt",
    min_samples_leaf=1,
    class_weight="balanced",
    random_state=RANDOM_STATE,
    n_jobs=1,
)
random_forest.fit(X_train, y_train)
print("Random Forest ajustada com 250 árvores.")

## 5. Avaliação no conjunto de teste

Acurácia sozinha pode esconder desempenho fraco na classe minoritária. Por isso, além da acurácia, são reportados **precisão**, **recall**, **F1 macro** (que dá o mesmo peso a tintos e brancos) e **ROC-AUC**. O requisito mínimo da atividade é acurácia acima de 85%; a tabela a seguir permite verificar essa condição diretamente em dados não usados no ajuste.

In [ ]:
modelos = {
    "KNN (otimizado por CV)": busca_knn.best_estimator_,
    "Random Forest": random_forest,
}
resultados = []
predicoes = {}

for nome, modelo in modelos.items():
    y_pred = modelo.predict(X_test)
    y_prob = modelo.predict_proba(X_test)[:, 1]
    predicoes[nome] = y_pred
    resultados.append({
        "modelo": nome,
        "acurácia": accuracy_score(y_test, y_pred),
        "precisão_macro": precision_score(y_test, y_pred, average="macro"),
        "recall_macro": recall_score(y_test, y_pred, average="macro"),
        "f1_macro": f1_score(y_test, y_pred, average="macro"),
        "roc_auc": roc_auc_score(y_test, y_prob),
    })

tabela_resultados = (pd.DataFrame(resultados)
                      .sort_values(["acurácia", "f1_macro"], ascending=False)
                      .reset_index(drop=True))
tabela_exibicao = tabela_resultados.copy()
for coluna in tabela_exibicao.columns[1:]:
    tabela_exibicao[coluna] = tabela_exibicao[coluna].map(lambda valor: f"{valor:.2%}")
display(tabela_exibicao)

assert (tabela_resultados["acurácia"] > 0.85).all(), "Algum modelo ficou abaixo da meta de 85%."
print("✓ Ambos os modelos superaram a meta de 85% de acurácia no conjunto de teste.")

In [ ]:
# Matrizes de confusão e relatórios por classe.
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for ax, (nome, y_pred) in zip(axes, predicoes.items()):
    ConfusionMatrixDisplay.from_predictions(
        y_test,
        y_pred,
        display_labels=encoder.classes_,
        cmap="Blues",
        colorbar=False,
        ax=ax,
    )
    ax.set_title(nome)
fig.suptitle("Matrizes de confusão no conjunto de teste", y=1.03, fontsize=14)
plt.tight_layout()
plt.show()

for nome, y_pred in predicoes.items():
    print(f"\n### {nome}")
    display(pd.DataFrame(classification_report(
        y_test, y_pred, target_names=encoder.classes_, output_dict=True
    )).T.round(3))

In [ ]:
# Interpretabilidade do Random Forest: quais medições químicas mais ajudaram?
importancia = (pd.Series(random_forest.feature_importances_, index=feature_cols)
                 .sort_values(ascending=True))
fig, ax = plt.subplots(figsize=(8, 5))
importancia.tail(10).plot(kind="barh", ax=ax, color="#2874A6")
ax.set_title("10 maiores importâncias — Random Forest")
ax.set_xlabel("Importância por redução de impureza")
ax.set_ylabel("Variável físico-química")
plt.tight_layout()
plt.show()
display(importancia.sort_values(ascending=False).rename("importância").to_frame().round(4))

## 6. Discussão dos resultados

A tabela de métricas e as matrizes de confusão devem ser lidas em conjunto:

- A meta obrigatória é atendida se a coluna **acurácia** for maior que 85% para os dois modelos.
- Se o Random Forest liderar, a interpretação provável é que os limites e interações entre medidas químicas sejam não lineares — exatamente o tipo de padrão em que conjuntos de árvores costumam se destacar.
- Se o KNN se aproximar do Random Forest, isso indica que as classes já formam grupos bem separados após a padronização. O `StandardScaler` é indispensável aqui: sem ele, variáveis medidas em dezenas ou centenas dominariam a distância.
- O F1 macro e o recall por classe confirmam se o resultado não veio apenas da maioria de vinhos brancos. Como o particionamento foi estratificado, essa comparação é justa.
- A importância da Random Forest sugere quais substâncias contribuem para a separação, mas **não prova causalidade**. Ela também pode dividir importância entre variáveis correlacionadas.

**Limitações:** esta é uma base de vinho verde português; não se deve generalizar automaticamente para todos os países, uvas ou processos produtivos. Além disso, as medições disponíveis podem ser feitas em laboratório, portanto o modelo não substitui uma avaliação completa de produção ou qualidade.

In [ ]:
# Escolha objetiva do melhor modelo: maior acurácia; em empate, maior F1 macro.
ranking = tabela_resultados.sort_values(["acurácia", "f1_macro"], ascending=False)
melhor_nome = ranking.iloc[0]["modelo"]
melhor_modelo = modelos[melhor_nome]

ARTEFATO_DIR = ROOT_DIR / "outputs"
ARTEFATO_DIR.mkdir(exist_ok=True)
caminho_modelo = ARTEFATO_DIR / "melhor_modelo_tipo_vinho.joblib"
pacote_modelo = {
    "model": melhor_modelo,
    "feature_names": feature_cols,
    "target_classes": encoder.classes_.tolist(),
    "test_accuracy": float(ranking.iloc[0]["acurácia"]),
    "dataset_source": source_url,
    "random_state": RANDOM_STATE,
}
joblib.dump(pacote_modelo, caminho_modelo)

print(f"Melhor modelo: {melhor_nome}")
print(f"Acurácia de teste: {ranking.iloc[0]['acurácia']:.2%}")
print(f"Modelo salvo em: {caminho_modelo.resolve()}")

In [ ]:
# Demonstração de reutilização do artefato salvo com uma observação nunca vista.
artefato_carregado = joblib.load(caminho_modelo)
exemplo = X_test.iloc[[0]]
codigo_previsto = artefato_carregado["model"].predict(exemplo)[0]
classe_prevista = artefato_carregado["target_classes"][codigo_previsto]
classe_real = encoder.inverse_transform([y_test[0]])[0]

print("Previsão para a primeira observação do teste:")
print(f"  Tipo real:     {classe_real}")
print(f"  Tipo previsto: {classe_prevista}")

## Conclusão

A análise evidenciou diferenças químicas consistentes entre vinhos tintos e brancos, especialmente em acidez volátil, açúcar residual e dióxido de enxofre. Ao restringir os modelos às 11 variáveis físico-químicas e reservar 20% dos dados para teste, a avaliação mede generalização em vez de memorização. KNN e Random Forest são então comparados de forma reproduzível, com o melhor modelo persistido em `outputs/melhor_modelo_tipo_vinho.joblib`.

### Pontos para a arguição oral

1. **Por que criar `tipo_vinho`?** Porque os dois arquivos oficiais representam tipos distintos e a atividade permite construir a variável alvo quando ela não está em uma única coluna. A criação é documentada e não usa uma regra arbitrária sobre os números.
2. **Por que excluir `quality`?** Ela é uma nota sensorial, enquanto o objetivo é classificar o tipo apenas pelas medidas químicas. Isso torna a pergunta mais rigorosa e evita depender de uma avaliação posterior.
3. **Por que 80/20 e `stratify`?** 80% fornecem treinamento suficiente; 20% simulam dados novos. `stratify` conserva a proporção das classes em ambos os conjuntos.
4. **Por que escalar somente no KNN?** Distâncias são sensíveis à unidade de medida; árvores tomam decisões por limiares e não precisam de padronização.
5. **Por que olhar F1 macro além da acurácia?** A base é desbalanceada. O F1 macro impede que um bom desempenho apenas nos brancos esconda falhas nos tintos.